# 00 — Session bootstrap (smoke test)

**What this notebook does.** Prepares a fresh Colab or Kaggle session for
this project: reads the GitHub token from the host secret store, clones (or
hard-resets) the repo, installs the missing dependencies, mounts Drive on
Colab, resolves and verifies `DATA_ROOT`, and wires `outputs/` and
`checkpoints/` into `PERSISTENT_DIR` so a killed session loses nothing. It
then lists what is actually in `DATA_ROOT` and verifies the session.

**What must already exist.**

- a GitHub fine-grained PAT with *Contents: read and write* on this repo,
  stored as a host secret named exactly `GH_TOKEN`
  (Colab: key icon in the sidebar, notebook access ON —
  Kaggle: Add-ons → Secrets, attached to this notebook)
- the dataset folders uploaded to the location named in
  `configs/default.yaml` under `session.colab.data_root` (Colab) or
  `session.kaggle.dataset_slug` (Kaggle)

**What it produces.** Nothing committed — it is a smoke test. It leaves the
session ready: `PATHS` in memory, the repo importable, `PERSISTENT_DIR`
created, `data/`, `outputs/` and `checkpoints/` symlinked inside the repo.

**Expected runtime on a free T4.** Under 2 minutes: ~30 s for the Drive
mount consent on Colab, ~30–60 s for the pip install the first time, a few
seconds for everything else. Re-runs are faster because every step is
idempotent.

**This notebook is the template.** Every other notebook copies cell 1 below
verbatim, and ends with the same checks-then-push pair.

## Cell 1 — the standard bootstrap block

This is the only cell that may differ from the repo's own code, and it is
identical in every notebook. It exists because of a chicken-and-egg problem:
the repo is not on the host yet, so the code that clones it has to be fetched
first. It reads `GH_TOKEN` from the host secret store (never a literal),
downloads `scripts/bootstrap_session.py` through the GitHub API with that
token — which works whether the repo is public or private — then hands over
to `bootstrap()`, which does everything else and returns `PATHS`.

The token is deleted from the namespace immediately after use so it cannot
be printed by a later cell or captured in a saved output. Re-running this
cell after a disconnect is safe and is the correct way to recover.

In [ ]:
# --- standard bootstrap block: identical in every notebook ---------------
OWNER, REPO, BRANCH = "arhorri", "boundary", "main"

import importlib, os, pathlib, sys, urllib.request


def _gh_token():
    """Read GH_TOKEN from whichever secret store this host provides."""
    try:
        from google.colab import userdata

        return userdata.get("GH_TOKEN")
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret("GH_TOKEN")
    except Exception:
        pass
    return os.environ.get("GH_TOKEN")


_token = _gh_token()
if not _token:
    raise SystemExit(
        "GH_TOKEN secret is missing.\n"
        "  Colab : key icon in the left sidebar -> add GH_TOKEN -> notebook access ON\n"
        "  Kaggle: Add-ons -> Secrets -> add GH_TOKEN -> attach to this notebook"
    )

_req = urllib.request.Request(
    f"https://api.github.com/repos/{OWNER}/{REPO}/contents/scripts/bootstrap_session.py?ref={BRANCH}",
    headers={
        "Authorization": f"Bearer {_token}",
        "Accept": "application/vnd.github.raw",
    },
)
pathlib.Path("bootstrap_session.py").write_bytes(urllib.request.urlopen(_req).read())
del _token

if str(pathlib.Path.cwd()) not in sys.path:
    sys.path.insert(0, str(pathlib.Path.cwd()))
import bootstrap_session

bootstrap_session = importlib.reload(bootstrap_session)

PATHS = bootstrap_session.bootstrap(
    repo_url=f"https://github.com/{OWNER}/{REPO}.git", branch=BRANCH
)

## Cell 2 — what is actually in DATA_ROOT

The bootstrap has already asserted that every folder named in
`session.expected_datasets` is present. This cell shows *what is inside them*,
because the five datasets are packaged differently from each other and later
steps must not assume a shared layout. Read the per-dataset subfolder names
and file extensions here: they are the input to the dataset audit in step 1,
which is what decides MODE A vs MODE B per folder.

Nothing is loaded or decoded — this only walks the directory tree.

In [ ]:
from pathlib import Path

data_root = Path(PATHS["data_root"])
print(f"DATA_ROOT: {data_root}\n")

for dataset in sorted(p for p in data_root.iterdir() if p.is_dir()):
    files = [f for f in dataset.rglob("*") if f.is_file()]
    exts = sorted({f.suffix.lower() or "(none)" for f in files})
    subdirs = sorted(p.name for p in dataset.iterdir() if p.is_dir())
    print(f"{dataset.name:<12} {len(files):>6} files   ext: {', '.join(exts[:8])}")
    if subdirs:
        print(f"{'':<12} subfolders: {', '.join(subdirs[:12])}")
    for sample in files[:2]:
        print(f"{'':<12} e.g. {sample.relative_to(dataset)}")

## Cell 3 — checks

Nothing in this project is run on a local machine, so correctness has to be
established here. Every notebook ends with a cell like this one: it asserts
that the step's output is well-formed and prints `PASS` or `FAIL` per check,
then raises if anything failed, so a broken session cannot be mistaken for a
working one just because the cells ran.

For this notebook the claim being verified is: *this session is usable*. That
means the platform was recognised, every expected dataset is present and
non-empty, `PERSISTENT_DIR` is writable, the repo-relative `outputs/` and
`checkpoints/` really point into it, `src/` imports, and a GPU is attached.

In [ ]:
import os
from pathlib import Path

checks = []


def check(name, ok, detail=""):
    checks.append((name, bool(ok)))
    print(f"{'PASS' if ok else 'FAIL'}  {name}{'  -- ' + detail if detail else ''}")


repo_root = Path(PATHS["repo_root"])
data_root = Path(PATHS["data_root"])
persistent_dir = Path(PATHS["persistent_dir"])

check("platform is a remote host", PATHS["platform"] in ("colab", "kaggle"),
      PATHS["platform"])
check("repo checkout present", (repo_root / ".git").is_dir(), str(repo_root))

from src.paths import match_datasets, resolve_paths

check("src/ importable", callable(resolve_paths))

found_as, missing = match_datasets(data_root, PATHS["expected_datasets"])
check("every expected dataset present", not missing,
      f"missing: {missing}" if missing else f"{len(found_as)} folders")

for expected, actual in sorted(found_as.items()):
    n = sum(1 for f in (data_root / actual).rglob("*") if f.is_file())
    check(f"{expected} non-empty", n > 0, f"{n} files")

probe = persistent_dir / ".bootstrap_write_probe"
try:
    probe.write_text("ok")
    probe.unlink()
    writable = True
except Exception as exc:
    writable = False
    print(f"      write probe failed: {exc}")
check("PERSISTENT_DIR writable", writable, str(persistent_dir))

for name in ("outputs", "checkpoints"):
    link = repo_root / name
    inside = link.exists() and str(Path(os.path.realpath(link))).startswith(
        str(persistent_dir.resolve())
    )
    check(f"{name}/ persists across session death", inside,
          os.path.realpath(link) if link.exists() else "missing")

import torch

check("GPU attached", torch.cuda.is_available(), PATHS["gpu"])

failed = [n for n, ok in checks if not ok]
print(f"\n{len(checks) - len(failed)}/{len(checks)} checks passed")
if failed:
    raise AssertionError("failed checks: " + ", ".join(failed))

## Cell 4 — push generated results back to the repo

Every notebook ends here. Results produced on the host are worthless to the
next step until they are committed, so `push_results` stages `reports/`,
`configs/` and `notebooks/` only — never data, checkpoints or outputs — and
pushes with the same `GH_TOKEN` secret the bootstrap used.

This notebook generates no reports, so the expected result is the printed
no-op message. Later notebooks write into `reports/` before this cell and it
will actually commit.

In [ ]:
from scripts.push_results import push_results

push_results("step 0b: bootstrap smoke test", paths=PATHS)